# PyTorch: Binary classification using FastAI

Dataset: cats vs dogs

## Load Dataset

In [ ]:
import numpy as np
import shutil
from pathlib import Path
from common import CV_DATA_DIR

In [ ]:
from fastai.data.block import DataBlock, CategoryBlock
from fastai.data.transforms import RandomSplitter,  parent_label
from fastai.vision.augment import Resize, ResizeMethod, RandomResizedCrop, aug_transforms
from fastai.vision.data import ImageBlock, ImageGetter
from fastai.vision.widgets import ImageClassifierCleaner
from fastai.vision.learner import vision_learner, load_learner
from fastai.metrics import error_rate
from fastai.vision.models import resnet18
from fastai.interpret import ClassificationInterpretation

In [ ]:
DATA_DIR=CV_DATA_DIR/"animals"/"cats-and-dogs"

In [ ]:
db = DataBlock(
    blocks=[ImageBlock, CategoryBlock],
    get_items=ImageGetter(folders=["Dog", "Cat"]),
    splitter=RandomSplitter(valid_pct=0.2, seed=13),
    get_y=parent_label,
)

## Transformation

In [ ]:
db = db.new(item_tfms=[Resize(size=224, method=ResizeMethod.Pad, pad_mode="zeros")])
loaders = db.dataloaders(DATA_DIR)
loaders.train.show_batch(max_n=4, nrows=1)

In [ ]:
db = db.new(item_tfms=[RandomResizedCrop(size=224, min_scale=0.3)])
loaders = db.dataloaders(DATA_DIR)
loaders.train.show_batch(max_n=4, nrows=1, unique=True)

## Data Augmentation

In [ ]:
db = db.new(
    item_tfms=[Resize(size=224)],
    batch_tfms=aug_transforms(pad_mode="zeros", mult=2, min_scale=0.3))
loaders = db.dataloaders(DATA_DIR)
loaders.train.show_batch(max_n=4, nrows=1, unique=True)

## Training

In [ ]:
db = db.new(
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms())
loaders = db.dataloaders(DATA_DIR)

In [ ]:
learn = vision_learner(loaders, resnet18, metrics=error_rate)

In [ ]:
learn.fine_tune(4)

## Evaluate

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)

In [ ]:
# Plot confusion matrix
interp.plot_confusion_matrix(figsize=(10,5))

In [ ]:
# Get top losses samples
interp.plot_top_losses(5, nrows=1)

## Cleaning dataset

In [ ]:
cleaner = ImageClassifierCleaner(learn)
cleaner

In [ ]:
# Delete files
for idx in cleaner.delete(): cleaner.fns[idx].unlink()

In [ ]:
# Move files to correct subfolder (category)
for idx,cat in cleaner.change(): shutil.move(str(cleaner.fns[idx]), DATA_DIR/cat)

## Export

In [ ]:
EXPORT_PATH = Path("files")/"fastai_basic"
EXPORT_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
learn.export(EXPORT_PATH/"fastai.pkl")

## Import

In [ ]:
# Load from exports
learn_inf = load_learner(EXPORT_PATH/"fastai.pkl")

In [ ]:
# Get learner vocabulary
vocab = learn_inf.dls.vocab

In [ ]:
# Predict
pred, pred_idx, probs = learn_inf.predict(DATA_DIR/"Dog"/"0.jpg")

print("Result:")
print("...pred:", pred)
print("...idx:", int(pred_idx))
print("...prob:", np.max(probs.numpy()))